# 04 Churn Model

Build a churn label, compare models, and inspect feature importance.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.data_processing import load_events
from src.modeling import build_churn_dataset, train_models
from src.visualization import plot_feature_importance

EVENTS_PATH = "C:/Users/chuan/OneDrive/Desktop/Intern_Prepare/业务准备/ecommerce-user-retention-analysis/data/events.csv"
OUTPUT_DIR = "C:/Users/chuan/OneDrive/Desktop/Intern_Prepare/业务准备/ecommerce-user-retention-analysis/outputs/figures"

df = load_events(EVENTS_PATH)
ds, anchor_date, feature_start, max_date = build_churn_dataset(df, feature_window_days=14, churn_window_days=14)

print("Feature window:", feature_start.date(), "to", anchor_date.date())
print("Label window:", (anchor_date + pd.Timedelta(days=1)).date(), "to", max_date.date())

model_result, best_model_name, best_model, test_pred, report, importance_df = train_models(ds)
model_result

Feature window: 2015-08-22 to 2015-09-04
Label window: 2015-09-05 to 2015-09-18


,model,auc,average_precision,precision,recall,f1,best_threshold
0,gbdt,0.695987,0.977349,0.961995,0.999293,0.980290,0.4
1,logistic_regression,0.694669,0.977171,0.964476,0.993494,0.978770,0.1
2,random_forest,0.685194,0.975763,0.962983,0.997910,0.980135,0.1
3,lightgbm,0.661786,0.973021,0.963457,0.996614,0.979755,0.1


In [3]:
print("Best model:", best_model_name)
print(report)

Best model: gbdt
              precision    recall  f1-score   support

           0       0.72      0.04      0.08      1404
           1       0.96      1.00      0.98     33968

    accuracy                           0.96     35372
   macro avg       0.84      0.52      0.53     35372
weighted avg       0.95      0.96      0.94     35372



In [4]:
if importance_df is not None:
    plot_feature_importance(importance_df, OUTPUT_DIR)
    display(importance_df.head(20))

,feature,importance
0,feat_lifetime_days,0.229398
1,feat_days_since_last_active,0.217376
2,feat_n_active_days,0.093027
3,w30_n_active_days,0.083121
4,w14_n_active_days,0.081373
5,w7_cnt_view,0.061486
6,w7_n_events,0.031139
7,feat_cnt_view,0.020940
8,w30_cnt_view,0.018488
9,w7_avg_hour,0.015892


In [6]:
test_pred.head()

,y_true,y_prob,y_pred
0,1,0.981068,1
1,1,0.970814,1
2,1,0.981068,1
3,1,0.974075,1
4,1,0.975852,1
